# Accelerating Vector Search: hipVS on AMD

## Importing libraries and setting up the environment

In [ ]:
import json
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import time
import gzip
import os
import torch
import pylibraft
from cuvs.neighbors import brute_force, ivf_flat, ivf_pq, cagra
from cuvs.common import Resources

import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp
import numpy as np

pylibraft.config.set_output_as(lambda device_ndarray: device_ndarray.copy_to_host())

if not torch.cuda.is_available():
  print("Warning: No GPU found. Please add GPU to your notebook")

In [ ]:
# Use the pool memory resource cuVS uses RMM allocator widely across its algorithms,
# including the performance-sensitive parts like IVF-PQ search. 
# It's strongly advised to set up the RMM pool memory resource to minimize the overheads
# of repeated HIP/CUDA allocations.


pool = rmm.mr.PoolMemoryResource(
    rmm.mr.CudaMemoryResource(),
    initial_pool_size=2**30
)
rmm.mr.set_current_device_resource(pool)
cp.cuda.set_allocator(rmm_cupy_allocator)

## Download and explore the data

In [3]:
# Download the dataset

def get_simplewiki_dataset(simplewiki_url, simplewiki_save_path):
    # Simple English Wikipedia dataset (about 170k articles)
    # Dataset articles will be splitted into paragraphs and encoded    
    try:        
        if not os.path.exists(simplewiki_save_path):
            util.http_get(simplewiki_url, simplewiki_save_path)    
    except Exception as e:
        print(f"Error downloading the datataset. {e}")
    else:
        print(f"Dataset saved to: {simplewiki_save_path}")



def create_and_encode_passages(simplewiki_save_path, encoder):
    # Extract paragraphs and creates passages to be encoded
    
    passages = []
    
    with gzip.open(simplewiki_save_path, 'rt', encoding='utf8') as f:
        for line in f:
            data = json.loads(line.strip())
            for paragraph in data['paragraphs']:
                # Passages as [title, text]
                passages.append([data['title'], paragraph])
        
    # Compute the embeddings from scratch (which can take a while depending on the GPU)
    corpus_embeddings = encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True)

    return passages, corpus_embeddings


# A helper for inspecting properties of an object
def show_properties(obj):
    return {
        attr: getattr(obj, attr)
        for attr in dir(obj)
        if type(getattr(type(obj), attr)).__name__ == 'getset_descriptor'
    }    

In [4]:
simplewiki_save_path = './data/simplewiki-2020-11-01.jsonl.gz'
simplewiki_url = 'http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz'

model_name = 'sentence-transformers/nq-distilbert-base-v1'
encoder = SentenceTransformer(model_name)

get_simplewiki_dataset(simplewiki_url, simplewiki_save_path)
passages, corpus_embeddings = create_and_encode_passages(simplewiki_save_path, encoder)

Dataset saved to: ./data/simplewiki-2020-11-01.jsonl.gz


Batches:   0%|          | 0/15927 [00:00<?, ?it/s]

## Data example

In [5]:
print(f'\nNumber of passages: {len(passages)}')
print(f'\nExample of passage:\n{passages[0]}')
print(f'\nExample of embedded passage:\n{corpus_embeddings[0][:10]}')


Number of passages: 509663

Example of passage:
['Ted Cassidy', 'Ted Cassidy (July 31, 1932 - January 16, 1979) was an American actor. He was best known for his roles as Lurch and Thing on "The Addams Family".']

Example of embedded passage:
tensor([-0.7203,  0.7746, -0.8595, -0.3508,  0.6317,  0.0244, -0.6441,  0.9293,
        -0.6116, -0.3703], device='cuda:0')


## Vector Search using AMD hipVS

### Brute Force KNN

In [6]:
# Resources  is a lightweight python wrapper around the corresponding
# C++ class of resources exposed by RAFT's C++ interface.
resources = Resources()

In [7]:
bf_index = brute_force.build(corpus_embeddings, metric='sqeuclidean', resources=resources)

# This function is asynchronous so we need to explicitly synchronize the GPU before we can measure the execution time
resources.sync()

In [8]:
query="What is creating tides?"
question_embedding = encoder.encode(query, convert_to_tensor=True)

In [9]:
%%time

top_k=5
distances, neighbors = brute_force.search(bf_index, question_embedding[None], top_k)

CPU times: user 36.5 ms, sys: 39.3 ms, total: 75.8 ms
Wall time: 74.1 ms


In [10]:
for k in range(top_k):
    print(f'Distance: {distances[0][k]}',f'Neighbor: {passages[neighbors[0][k]]}\n')

Distance: 94.91021728515625 Neighbor: ['Tide', "A tide is the periodic rising and falling of Earth's ocean surface caused mainly by the gravitational pull of the Moon acting on the oceans. Tides cause changes in the depth of marine and estuarine (river mouth) waters. Tides also make oscillating currents known as tidal streams (~'rip tides'). This means that being able to predict the tide is important for coastal navigation. The strip of seashore that is under water at high tide and exposed at low tide, called the intertidal zone, is an important ecological product of ocean tides."]

Distance: 159.54246520996094 Neighbor: ['Tidal energy', "Many things affect tides. The pull of the Moon is the largest effect, and most of the energy comes from the slowing of the Earth's spin."]

Distance: 159.74078369140625 Neighbor: ['Storm surge', 'A storm surge is a sudden rise of water hitting areas close to the coast. Storm surges are usually created by a hurricane or other tropical cyclone. The surg

### IVF-Flat

In [11]:
index_params = ivf_flat.IndexParams(n_lists=1024, 
                                    metric='sqeuclidean', 
                                    kmeans_n_iters=20,
                                    kmeans_trainset_fraction=0.5
                                   )
ivf_flat_index = ivf_flat.build(index_params, corpus_embeddings, resources=resources)
resources.sync()

In [12]:
# n_probes is the number of clusters we select in the first (coarse) search step.
# This is the only hyper parameter for search.
search_params = ivf_flat.SearchParams(n_probes=30)

query="What is creating tides?"
question_embedding = encoder.encode(query, convert_to_tensor=True)

In [13]:
%%time
# Search top 5 nearest neighbors.
top_k=5
distances, indices = ivf_flat.search(search_params, ivf_flat_index, question_embedding[None], k=top_k,)

CPU times: user 54.8 ms, sys: 4.28 ms, total: 59.1 ms
Wall time: 56.9 ms


In [14]:
for k in range(top_k):
    print(f'Distance: {distances[0][k]}',f'Neighbor: {passages[indices[0][k]]}\n')

Distance: 94.91002655029297 Neighbor: ['Tide', "A tide is the periodic rising and falling of Earth's ocean surface caused mainly by the gravitational pull of the Moon acting on the oceans. Tides cause changes in the depth of marine and estuarine (river mouth) waters. Tides also make oscillating currents known as tidal streams (~'rip tides'). This means that being able to predict the tide is important for coastal navigation. The strip of seashore that is under water at high tide and exposed at low tide, called the intertidal zone, is an important ecological product of ocean tides."]

Distance: 159.5424041748047 Neighbor: ['Tidal energy', "Many things affect tides. The pull of the Moon is the largest effect, and most of the energy comes from the slowing of the Earth's spin."]

Distance: 159.7407989501953 Neighbor: ['Storm surge', 'A storm surge is a sudden rise of water hitting areas close to the coast. Storm surges are usually created by a hurricane or other tropical cyclone. The surge 

### IVF-Flat PQ

In [15]:
pq_dim = 1
while pq_dim * 2 < corpus_embeddings.shape[1]:
    pq_dim = pq_dim * 2

index_params = ivf_pq.IndexParams(n_lists=1024, metric='sqeuclidean', pq_dim=pq_dim)
index = ivf_pq.build(index_params, corpus_embeddings, resources=resources)

resources.sync()

using ivf_pq::index_params nrows 509663, dim 768, n_lists 1024, pq_dim 512


In [16]:
search_params = ivf_pq.SearchParams()
show_properties(search_params)

{'internal_distance_dtype': 0, 'lut_dtype': 0, 'n_probes': 20}

In [17]:
query="What is creating tides?"
question_embedding = encoder.encode(query, convert_to_tensor=True)

In [18]:
%%time
top_k=5
distances, neighbors = ivf_pq.search(search_params, index, question_embedding[None], top_k, resources=resources)

CPU times: user 51.5 ms, sys: 16.5 ms, total: 68 ms
Wall time: 66.4 ms


In [19]:
for k in range(top_k):
    print(f'Distance: {distances[0][k]}',f'Neighbor: {passages[neighbors[0][k]]}\n')

Distance: 94.8750991821289 Neighbor: ['Tide', "A tide is the periodic rising and falling of Earth's ocean surface caused mainly by the gravitational pull of the Moon acting on the oceans. Tides cause changes in the depth of marine and estuarine (river mouth) waters. Tides also make oscillating currents known as tidal streams (~'rip tides'). This means that being able to predict the tide is important for coastal navigation. The strip of seashore that is under water at high tide and exposed at low tide, called the intertidal zone, is an important ecological product of ocean tides."]

Distance: 158.55657958984375 Neighbor: ['Storm surge', 'A storm surge is a sudden rise of water hitting areas close to the coast. Storm surges are usually created by a hurricane or other tropical cyclone. The surge happens because a storm has fast winds and low atmospheric pressure. Water is pushed on shore, and the water level rises. Strong storm surges can flood coastal towns and destroy homes. A storm sur

## CAGRA

In [20]:
# Set the index parameters
build_params = cagra.IndexParams(metric="sqeuclidean")

# Build the index
index = cagra.build(build_params, corpus_embeddings,  resources=resources)
resources.sync()

# Set the search parameters
search_params = cagra.SearchParams()
show_properties(search_params)

using ivf_pq::index_params nrows 509663, dim 768, n_lists 713, pq_dim 192


[2025-08-11 20:58:33.511] [RAFT] [info] optimizing graph
[2025-08-11 20:58:34.176] [RAFT] [info] Graph optimized, creating index


{'algo': 3,
 'hashmap_max_fill_rate': 0.5,
 'hashmap_min_bitlen': 0,
 'hashmap_mode': 2,
 'itopk_size': 64,
 'max_iterations': 0,
 'max_queries': 0,
 'min_iterations': 0,
 'num_random_samplings': 1,
 'rand_xor_mask': 1213332,
 'search_width': 1,
 'team_size': 0,
 'thread_block_size': 0}

In [21]:
# Encoding the query
query="What is creating tides?"
question_embedding = encoder.encode(query, convert_to_tensor=True)

In [22]:
%%time
# Search and return the top five closest elements 
top_k=5
distances, neighbors = cagra.search(search_params, index, question_embedding[None],
                                     top_k)

CPU times: user 7.95 ms, sys: 3.99 ms, total: 11.9 ms
Wall time: 10.8 ms


In [23]:
for k in range(top_k):
    print(f'Distance: {distances[0][k]}',f'Neighbor: {passages[neighbors[0][k]]}\n')

Distance: 94.91004180908203 Neighbor: ['Tide', "A tide is the periodic rising and falling of Earth's ocean surface caused mainly by the gravitational pull of the Moon acting on the oceans. Tides cause changes in the depth of marine and estuarine (river mouth) waters. Tides also make oscillating currents known as tidal streams (~'rip tides'). This means that being able to predict the tide is important for coastal navigation. The strip of seashore that is under water at high tide and exposed at low tide, called the intertidal zone, is an important ecological product of ocean tides."]

Distance: 159.54251098632812 Neighbor: ['Tidal energy', "Many things affect tides. The pull of the Moon is the largest effect, and most of the energy comes from the slowing of the Earth's spin."]

Distance: 159.74075317382812 Neighbor: ['Storm surge', 'A storm surge is a sudden rise of water hitting areas close to the coast. Storm surges are usually created by a hurricane or other tropical cyclone. The surg